# Calibration Dataset Construction

This notebook constructs a small calibration dataset consisting of ten OCT
B-scans.

Five scans are selected from fast progressors and five from slow
progressors.

For every scan the notebook performs:

1. Heidelberg E2E loading.
2. Anatomical preprocessing.
3. Structural-HyperTD detection.
4. Visualization of the preprocessed scan.
5. Visualization of the detected barcode intervals.

These images will subsequently be manually annotated and used to tune the
detector parameters.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from src.detector.data import build_grouped_volume_registry, load_e2e_volume
from src.detector.preprocessing import preprocess_bscan
from src.detector.detector import detect_structural_hypertransmission

print( "Project root:",PROJECT_ROOT)

In [ ]:
E2E_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "heyex"
    / "meta"
)

registry = build_grouped_volume_registry(
    e2e_directory=E2E_DIRECTORY,
    progression_groups={
        "fast": [8, 9, 12, 41, 49],
        "slow": [17, 23, 35, 36, 47],
    },
)

In [ ]:
SCAN_CONFIG = {
    "fast": {
        8: 48,
        9: 85,
        12: 38,
        41: 59,
        49: 33,
    },

    "slow": {
        17: 55,
        23: 43,
        35: 38,
        36: 59,
        47: 31,
    },
}

In [ ]:
PREPROCESSING_CONFIG = {
    # Retinal layer used for anatomical alignment.
    "layer_name": "BM",

    # Flattening
    # Target row for the selected retinal layer after flattening.
    #   None: automatically use the median layer position.
    #   Larger values move the flattened layer lower in the image.
    "reference_row": None,

    # Pixel value assigned to regions created after vertical shifting.
    #   Usually left at zero.
    "flatten_fill_value": 0.0,


    # Sub-layer crop
    # Number of pixels retained beneath the selected retinal layer.
    #   +: Includes more deep tissue and choroid.
    #   -: Restricts analysis to tissue immediately below the layer.
    "depth_below_layer": 150,

    # Whether the reference layer itself is included in the crop.
    #   False begins one pixel below the layer.
    "include_boundary": True,

    # T: scans without sufficient depth are rejected.
    # F: shallow scans are padded.
    "require_full_depth": False,

    # Pixel value used when crop padding is required.
    "crop_fill_value": 0.0,


    # Intensity normalization method.
    #   "zscore"
    #   "minmax"
    #   "percentile"
    "normalization_method": "zscore",

    # Lower percentile used by percentile-based normalization.
    #   +: Ignores more dark outliers.
    #   -: Uses more of the darkest pixels.
    "lower_percentile": 1.0,

    # Upper percentile used by percentile-based normalization.
    #   +: Preserves more bright structures.
    #   -: Compresses very bright outliers.
    "upper_percentile": 99.0,


    # Denoising algorithm applied after normalization.
    "denoise_method": "gaussian",

    # Gaussian smoothing standard deviations (pixels).
    #   +: More smoothing, less speckle, but reduced spatial detail.
    #   -: Preserves fine structures but retains more noise.
    "gaussian_sigma": (
        1.0,   # depth (z)
        0.5,   # horizontal (x)
    ),
}

In [ ]:
DETECTOR_CONFIG = {
    # Pixel-level structural feature extraction

    # Standard deviation of the Gaussian smoothing applied before computing local image gradients.
    #   +: Reduces speckle noise and produces smoother verticality maps but may blur fine barcode structures.
    #   -: Preserves fine detail but makes verticality estimates more sensitive to noise.
    "verticality_smoothing_sigma": 1.0,

    # Minimum verticality required for a pixel to be considered structurally oriented.
    #   +: Keeps only strongly vertical structures.
    #   -: Includes weaker or noisier vertical structures.
    "verticality_threshold": 0.60,

    # Gradient-magnitude quantile used to define structural pixels.
    #   +: Produces a smaller structural mask by retaining only the strongest gradients.
    #   -: Produces a larger structural mask by excluding more tissue.
    "gradient_quantile": 0.80,

    # Minimum connected-component size (pixels) retained in the structural mask.
    #   +: Removes isolated structural detections.
    #   -: Retains smaller structures.
    #  Zero disables size filtering.
    "minimum_component_size": 0,

    # Upper intensity quantile measured within each cleaned column.
    #   +: emphasize the brightest hypertransmission pixels.
    #   -: Smaller values measure a broader portion of the intensity distribution.
    "column_upper_quantile": 0.90,

    # Minimum number of remaining pixels required after structural exclusion for a column to be considered reliable.
    #   +: Rejects more columns.
    #   -: Allows noisier columns to contribute.
    "minimum_valid_pixels": 5,

    # Gaussian smoothing applied to column-level feature signals.
    #   +: Produces smoother detector responses and reduces isolated peaks.
    #   -: Preserves rapid local changes.
    "signal_smoothing_sigma": 2.0,

    # Multiplier applied to the IQR when thresholding column medians.
    #   +: More conservative threshold.
    #   -: More sensitive threshold.
    "median_iqr_multiplier": 1.0,

    # Multiplier applied to the IQR when thresholding the column upper-intensity statistic.
    #   +: Requires stronger hypertransmission.
    #   -: Detects weaker hypertransmission.
    "q90_iqr_multiplier": 0.5,

    # Horizontal window width (pixels) used when evaluating local depth continuity.
    #   +: Produces smoother continuity estimates over larger regions.
    #   -: Responds more quickly to local changes.
    "continuity_window_width": 15,

    # Maximum vertical displacement (pixels) considered when matching adjacent image columns.
    #   +: Allows continuity across larger vertical shifts.
    #   -: Requires tighter alignment between neighboring columns.  
    "continuity_depth_lag": 4,

    # Small numerical constant preventing division by zero in nearly onstant image regions.
    "continuity_minimum_row_standard_deviation": 1e-6,

    # Quantile threshold applied to the continuity score.
    #   +: Requires stronger depth continuity.
    #   -: Accepts weaker continuity.
    "continuity_quantile": 0.60,

    # Quantile threshold applied to the fraction of vertically organized pixels within candidate columns.
    #   +: Retains only highly vertical candidate columns.
    #   -: Allows less organized candidates.
    "vertical_fraction_quantile": 0.70,

    # Minimum horizontal interval length retained after thresholding.
    #   +: Removes short detections.
    #   -: Preserves smaller barcode candidates.
    "minimum_positive_run": 5,

    # Largest negative gap (pixels) filled between neighboring detections.
    #   +: Merges nearby intervals.
    #   -: Keeps intervals separate.
    #   Zero disables gap filling.
    "maximum_negative_gap": 2,

    # Number of image columns ignored at each lateral edge before interval extraction.
    #   +: Suppresses more edge artefacts.
    #   -: Includes more peripheral image content.
    "edge_margin": 10,
}

In [ ]:
calibration_cases = {}

for progression_group, subject_scans in SCAN_CONFIG.items():

    for subject_id, bscan_index in subject_scans.items():

        matches = [
            record
            for record in registry
            if (
                int(record.subject_id) == int(subject_id)
                and str(record.progression_group).lower()
                == progression_group.lower()
            )
        ]

        if not matches:
            raise KeyError(
                f"Could not find subject {subject_id} "
                f"in group '{progression_group}'."
            )

        if len(matches) > 1:
            raise ValueError(
                f"Subject {subject_id} has multiple matching volumes. "
                "Resolve the exact E2E file before calibration."
            )

        record = matches[0]

        volume = load_e2e_volume(
            record.e2e_path
        )

        if not 0 <= bscan_index < len(volume):
            raise IndexError(
                f"B-scan {bscan_index} is outside the valid range "
                f"0 to {len(volume) - 1} for subject {subject_id}."
            )

        processed = preprocess_bscan(
            volume=volume,
            bscan_index=bscan_index,
            **PREPROCESSING_CONFIG,
        )

        detector_result = (
            detect_structural_hypertransmission(
                processed.denoised_scan,
                config=DETECTOR_CONFIG,
            )
        )

        calibration_cases[
            (
                progression_group,
                subject_id,
            )
        ] = {
            "record": record,
            "bscan_index": int(
                bscan_index
            ),
            "processed": processed,
            "detector": detector_result,
        }

        print(
            f"Completed {progression_group} "
            f"subject {subject_id}, "
            f"B-scan {bscan_index}"
        )

## Preprocessed Scans

In [ ]:
for (
    progression_group,
    subject_id,
), case in calibration_cases.items():

    processed = case["processed"]

    image = processed.denoised_scan

    plt.figure(figsize=(14,5))

    plt.imshow(
        image,
        cmap="gray",
        aspect="auto",
    )

    plt.title(
        f"Subject {subject_id} | "
        f"{progression_group.title()} | "
        f"B-scan {case['bscan_index']}"
    )

    plt.xlabel("Horizontal position")
    plt.ylabel("Depth below BM")

    plt.tight_layout()

    plt.show()

## Manual Ground-Truth Annotation

The same ten preprocessed scans are manually annotated below.

For each scan:

1. The denoised, preprocessed sub-BM scan is displayed.
2. A label is selected.
3. Horizontal regions are marked by clicking and dragging across the scan.
4. Annotations are displayed directly on the image.
5. The annotation can be saved as:
   - an annotated PNG for visual ground truth;
   - a JSON file containing the exact interval coordinates and labels.

These manual annotations will later be compared against the automatic detector
outputs for parameter tuning.

In [ ]:
%matplotlib widget

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

from matplotlib.widgets import (
    Button,
    RadioButtons,
    SpanSelector,
)

GROUND_TRUTH_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "manual_ground_truth"
)

GROUND_TRUTH_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Ground-truth output:",
    GROUND_TRUTH_DIRECTORY,
)

In [ ]:
class NotebookScanAnnotator:
    """
    Interactive horizontal interval annotator for one preprocessed OCT scan.

    Controls
    --------
    1. Select a label using the radio buttons.
    2. Drag horizontally across the OCT image.
    3. Repeat for additional intervals.
    4. Use Undo or Clear if necessary.
    5. Click Save Ground Truth to save:
       - annotated PNG
       - JSON interval annotations
    """

    LABEL_COLORS = {
        "Barcoding": "tab:red",
        "Early Atrophy (EA)": "tab:orange",
        "Normal": "tab:green",
        "Vessel / Structural": "tab:blue",
        "Uncertain": "tab:gray",
    }

    def __init__(
        self,
        image: np.ndarray,
        *,
        subject_id: int,
        progression_group: str,
        bscan_index: int,
        output_directory: str | Path,
        figure_size: tuple[float, float] = (
            14,
            6,
        ),
    ):
        self.image = np.asarray(
            image,
            dtype=np.float32,
        )

        if self.image.ndim != 2:
            raise ValueError(
                "image must be two-dimensional."
            )

        self.subject_id = int(
            subject_id
        )

        self.progression_group = str(
            progression_group
        )

        self.bscan_index = int(
            bscan_index
        )

        self.output_directory = Path(
            output_directory
        )

        self.output_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        self.figure_size = figure_size

        self.selected_label = (
            "Barcoding"
        )

        self.intervals = []

        self.interval_artists = []

        self.figure = None
        self.scan_axis = None
        self.control_axis = None

        self.radio_buttons = None
        self.undo_button = None
        self.clear_button = None
        self.save_button = None
        self.status_text = None
        self.span_selector = None

        self._build_interface()

    def _build_interface(
        self,
    ):
        self.figure = plt.figure(
            figsize=self.figure_size,
        )

        grid = (
            self.figure
            .add_gridspec(
                nrows=1,
                ncols=2,
                width_ratios=[
                    5.5,
                    1.3,
                ],
                wspace=0.12,
            )
        )

        self.scan_axis = (
            self.figure
            .add_subplot(
                grid[0, 0]
            )
        )

        self.control_axis = (
            self.figure
            .add_subplot(
                grid[0, 1]
            )
        )

        height, width = (
            self.image.shape
        )

        self.scan_axis.imshow(
            self.image,
            cmap="gray",
            aspect="auto",
            extent=(
                -0.5,
                width - 0.5,
                height - 0.5,
                -0.5,
            ),
        )

        self.scan_axis.set_title(
            f"Subject {self.subject_id} | "
            f"{self.progression_group.title()} | "
            f"B-scan {self.bscan_index}"
        )

        self.scan_axis.set_xlabel(
            "Horizontal position"
        )

        self.scan_axis.set_ylabel(
            "Depth below BM"
        )

        self.scan_axis.set_xlim(
            -0.5,
            width - 0.5,
        )

        self.scan_axis.set_ylim(
            height - 0.5,
            -0.5,
        )

        self.control_axis.set_title(
            "Ground truth"
        )

        self.control_axis.set_xticks([])
        self.control_axis.set_yticks([])

        # ----------------------------------------------
        # Label selector
        # ----------------------------------------------

        radio_axis = (
            self.control_axis
            .inset_axes(
                [
                    0.05,
                    0.56,
                    0.90,
                    0.36,
                ]
            )
        )

        self.radio_buttons = (
            RadioButtons(
                radio_axis,
                tuple(
                    self.LABEL_COLORS
                ),
                active=0,
            )
        )

        self.radio_buttons.on_clicked(
            self._set_label
        )

        # ----------------------------------------------
        # Undo
        # ----------------------------------------------

        undo_axis = (
            self.control_axis
            .inset_axes(
                [
                    0.10,
                    0.39,
                    0.80,
                    0.08,
                ]
            )
        )

        self.undo_button = Button(
            undo_axis,
            "Undo last",
        )

        self.undo_button.on_clicked(
            self._undo
        )

        # ----------------------------------------------
        # Clear
        # ----------------------------------------------

        clear_axis = (
            self.control_axis
            .inset_axes(
                [
                    0.10,
                    0.27,
                    0.80,
                    0.08,
                ]
            )
        )

        self.clear_button = Button(
            clear_axis,
            "Clear all",
        )

        self.clear_button.on_clicked(
            self._clear
        )

        # ----------------------------------------------
        # Save
        # ----------------------------------------------

        save_axis = (
            self.control_axis
            .inset_axes(
                [
                    0.10,
                    0.15,
                    0.80,
                    0.08,
                ]
            )
        )

        self.save_button = Button(
            save_axis,
            "Save Ground Truth",
        )

        self.save_button.on_clicked(
            self._save
        )

        # ----------------------------------------------
        # Status
        # ----------------------------------------------

        self.status_text = (
            self.control_axis
            .text(
                0.5,
                0.06,
                "Selected: Barcoding\n"
                "Drag horizontally on scan.",
                ha="center",
                va="center",
                transform=(
                    self.control_axis
                    .transAxes
                ),
            )
        )

        # ----------------------------------------------
        # Horizontal drag annotation
        # ----------------------------------------------

        self.span_selector = (
            SpanSelector(
                self.scan_axis,
                self._add_interval,
                direction="horizontal",
                useblit=True,
                interactive=False,
                drag_from_anywhere=False,
                props={
                    "alpha": 0.15,
                    "facecolor": (
                        "tab:red"
                    ),
                },
            )
        )

        self.figure.canvas.draw_idle()

    def _set_label(
        self,
        label: str,
    ):
        self.selected_label = str(
            label
        )

        self._set_status(
            f"Selected: {label}\n"
            "Drag horizontally on scan."
        )

    def _add_interval(
        self,
        x_min: float,
        x_max: float,
    ):
        width = self.image.shape[1]

        x_start = float(
            np.clip(
                min(
                    x_min,
                    x_max,
                ),
                0,
                width - 1,
            )
        )

        x_end = float(
            np.clip(
                max(
                    x_min,
                    x_max,
                ),
                0,
                width - 1,
            )
        )

        if x_end <= x_start:
            return

        interval = {
            "label": (
                self.selected_label
            ),
            "x_start": x_start,
            "x_end": x_end,
            "width_pixels": (
                x_end - x_start
            ),
        }

        self.intervals.append(
            interval
        )

        color = (
            self.LABEL_COLORS[
                self.selected_label
            ]
        )

        artist = (
            self.scan_axis
            .axvspan(
                x_start,
                x_end,
                color=color,
                alpha=0.30,
            )
        )

        self.interval_artists.append(
            artist
        )

        self._set_status(
            f"Added {self.selected_label}\n"
            f"x={x_start:.1f}–"
            f"{x_end:.1f}"
        )

    def _undo(
        self,
        _event=None,
    ):
        if not self.intervals:
            self._set_status(
                "Nothing to undo."
            )
            return

        self.intervals.pop()

        artist = (
            self.interval_artists
            .pop()
        )

        artist.remove()

        self._set_status(
            "Removed last interval."
        )

    def _clear(
        self,
        _event=None,
    ):
        for artist in (
            self.interval_artists
        ):
            artist.remove()

        self.intervals.clear()
        self.interval_artists.clear()

        self._set_status(
            "All intervals cleared."
        )

    def _save(
        self,
        _event=None,
    ):
        stem = (
            f"{self.progression_group}_"
            f"{self.subject_id:02d}_"
            f"bscan_"
            f"{self.bscan_index:03d}"
        )

        json_path = (
            self.output_directory
            / f"{stem}_ground_truth.json"
        )

        png_path = (
            self.output_directory
            / f"{stem}_ground_truth.png"
        )

        record = {
            "subject_id": (
                self.subject_id
            ),
            "progression_group": (
                self.progression_group
            ),
            "bscan_index": (
                self.bscan_index
            ),
            "image_shape": [
                int(value)
                for value
                in self.image.shape
            ],
            "annotations": (
                self.intervals
            ),
        }

        with json_path.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                record,
                file,
                indent=2,
            )

        # Save only the image panel,
        # including annotation overlays.
        extent = (
            self.scan_axis
            .get_window_extent()
            .transformed(
                self.figure
                .dpi_scale_trans
                .inverted()
            )
        )

        self.figure.savefig(
            png_path,
            dpi=300,
            bbox_inches=extent,
        )

        self._set_status(
            f"Saved {len(self.intervals)} intervals\n"
            f"{png_path.name}"
        )

        print(
            "Saved JSON:",
            json_path,
        )

        print(
            "Saved PNG:",
            png_path,
        )

    def _set_status(
        self,
        message: str,
    ):
        self.status_text.set_text(
            message
        )

        self.figure.canvas.draw_idle()

In [ ]:
manual_fast_08 = NotebookScanAnnotator(
    calibration_cases[
        ("fast", 8)
    ]["processed"].denoised_scan,
    subject_id=8,
    progression_group="fast",
    bscan_index=48,
    output_directory=(
        GROUND_TRUTH_DIRECTORY
    ),
)

In [ ]:
manual_fast_09 = NotebookScanAnnotator(
    calibration_cases[
        ("fast", 9)
    ]["processed"].denoised_scan,
    subject_id=9,
    progression_group="fast",
    bscan_index=85,
    output_directory=(
        GROUND_TRUTH_DIRECTORY
    ),
)

In [ ]:
manual_fast_12 = NotebookScanAnnotator(
    calibration_cases[
        ("fast", 12)
    ]["processed"].denoised_scan,
    subject_id=12,
    progression_group="fast",
    bscan_index=38,
    output_directory=(
        GROUND_TRUTH_DIRECTORY
    ),
)

In [ ]:
manual_fast_41 = NotebookScanAnnotator(
    calibration_cases[
        ("fast", 41)
    ]["processed"].denoised_scan,
    subject_id=41,
    progression_group="fast",
    bscan_index=59,
    output_directory=(
        GROUND_TRUTH_DIRECTORY
    ),
)

In [ ]:
manual_fast_49 = NotebookScanAnnotator(
    calibration_cases[
        ("fast", 49)
    ]["processed"].denoised_scan,
    subject_id=49,
    progression_group="fast",
    bscan_index=33,
    output_directory=(
        GROUND_TRUTH_DIRECTORY
    ),
)

In [ ]:
manual_slow_17 = NotebookScanAnnotator(
    calibration_cases[
        ("slow", 17)
    ]["processed"].denoised_scan,
    subject_id=17,
    progression_group="slow",
    bscan_index=55,
    output_directory=(
        GROUND_TRUTH_DIRECTORY
    ),
)

In [ ]:
manual_slow_23 = NotebookScanAnnotator(
    calibration_cases[
        ("slow", 23)
    ]["processed"].denoised_scan,
    subject_id=23,
    progression_group="slow",
    bscan_index=43,
    output_directory=(
        GROUND_TRUTH_DIRECTORY
    ),
)

In [ ]:
manual_slow_35 = NotebookScanAnnotator(
    calibration_cases[
        ("slow", 35)
    ]["processed"].denoised_scan,
    subject_id=35,
    progression_group="slow",
    bscan_index=38,
    output_directory=(
        GROUND_TRUTH_DIRECTORY
    ),
)

In [ ]:
manual_slow_36 = NotebookScanAnnotator(
    calibration_cases[
        ("slow", 36)
    ]["processed"].denoised_scan,
    subject_id=36,
    progression_group="slow",
    bscan_index=59,
    output_directory=(
        GROUND_TRUTH_DIRECTORY
    ),
)

In [ ]:
manual_slow_47 = NotebookScanAnnotator(
    calibration_cases[
        ("slow", 47)
    ]["processed"].denoised_scan,
    subject_id=47,
    progression_group="slow",
    bscan_index=31,
    output_directory=(
        GROUND_TRUTH_DIRECTORY
    ),
)